# BME 574 — Project 5
## Can your generative model beat two lines of code?

A variational autoencoder gives every example a score: the ELBO, which is a **lower
bound** on how probable the model thinks that example is. Score low enough and we call
the example an anomaly. This is how VAEs are used for quality control, novelty detection
and "find me the unusual patients" in a great deal of published biomedical work.

Your job is to build one, score with it honestly, and then find out whether it was worth
building.

**What you will hand in**

1. This notebook, run end to end, with all five checks passing.
2. A one-page memo answering the six questions in §7 of the spec.
3. `AI_USE.md`.

**What is graded.** Not your AUC. The marks are for a defensible comparison: the model
trained only on normal cases, the scaler fitted only on training rows, the baselines run
honestly, the spread across seeds reported, and the conclusion stated even when it is
"my model lost".

> One of the tracks in this project is one where the VAE **does not beat** the trivial
> baselines. Nobody will tell you which. Reporting that convincingly earns full marks.

In [ ]:
import json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, roc_curve

torch.set_num_threads(4)
TRACK = 'thyroid'          # <-- your track, as assigned
BUNDLE = 'bundles_p5/%s.npz' % TRACK
print('track:', TRACK)

---
## 1 · The data, and the one rule that matters

The bundle holds every case, a label, and three index arrays. The split is **not** an
ordinary train/val/test split:

| split | contains |
|---|---|
| `train_idx` | normal cases **only** — the model never sees an anomaly while learning |
| `val_idx` | normal + abnormal — for choosing anything you need to choose |
| `test_idx` | normal + abnormal — scored once, at the end |

That is what makes this unsupervised anomaly detection rather than classification: the
training signal contains no information about what an anomaly looks like.

**The rule.** Anything fitted — the scaler, the PCA, the covariance for Mahalanobis, the
VAE itself — is fitted on `train_idx` and nothing else. A scaler fitted on all the rows
has already told your model where the anomalies are.

In [ ]:
d = np.load(BUNDLE, allow_pickle=True)
X, y = d['X'], d['y']
normal = int(d['normal'])
tr, va, te = d['train_idx'], d['val_idx'], d['test_idx']
class_names = [str(s) for s in d['class_names']]
data_kind = str(d['data_kind'])

print('source :', str(d['source']))
print('notes  :', str(d['notes']))
print()
print('%d cases, %d features' % X.shape)
print('normal class: %r' % class_names[normal])
for c in sorted(set(y.tolist())):
    if (y == c).sum():
        print('   %-28s %5d  %s' % (class_names[c], (y == c).sum(),
                                    'NORMAL' if c == normal else 'anomaly'))
print()
print('train %5d  (abnormal in train: %d)' % (len(tr), (y[tr] != normal).sum()))
print('val   %5d  (%d abnormal)' % (len(va), (y[va] != normal).sum()))
print('test  %5d  (%d abnormal)' % (len(te), (y[te] != normal).sum()))

In [ ]:
# Written for you: standardise using the TRAINING NORMALS only.
mu_, sd_ = X[tr].mean(0), X[tr].std(0) + 1e-8
Z = ((X - mu_) / sd_).astype(np.float32)
Ztr, Zva, Zte = Z[tr], Z[va], Z[te]

lab_va = (y[va] != normal).astype(int)      # 1 = anomaly
lab_te = (y[te] != normal).astype(int)
print('scaled with the training normals: mean %.3f, sd %.3f on train'
      % (Ztr.mean(), Ztr.std()))
print('test prevalence of anomalies: %.3f' % lab_te.mean())

---
## 2 · TODO 1 — build the model

A VAE is an encoder that outputs a **distribution**, and a decoder that reads a sample
from it.

- the encoder body maps the input to a hidden representation;
- **two separate heads** read that body: one gives $\mu$, the other $\log\sigma^2$;
- the decoder maps $z$ back to the input width.

Two mistakes to avoid, both of which run without complaint:

- putting an activation on `mu` or `logvar` — they must be free to take any real value,
  and squashing `logvar` is the usual way people accidentally fix the variance;
- making the decoder end in a `ReLU` or a `Sigmoid`. Your data is standardised, so it is
  negative about half the time.

In [ ]:
class VAE(nn.Module):
    def __init__(self, d, zdim, hidden=(128, 64), seed=0):
        super().__init__()
        torch.manual_seed(seed)
        # ---- TODO 1 ------------------------------------------------------------
        # Build four attributes:
        #   self.enc      Linear(d, hidden[0]) -> ReLU -> Linear(hidden[0], hidden[1]) -> ReLU
        #   self.mu       Linear(hidden[1], zdim)
        #   self.logvar   Linear(hidden[1], zdim)
        #   self.dec      Linear(zdim, hidden[1]) -> ReLU -> Linear(hidden[1], hidden[0])
        #                 -> ReLU -> Linear(hidden[0], d)
        raise SystemExit('TODO 1: build self.enc, self.mu, self.logvar and self.dec')
        # ---- end TODO 1 --------------------------------------------------------

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        return self.dec(z), mu, logvar

In [ ]:
# CHECK 1
_m = VAE(Ztr.shape[1], 8, seed=0)
_x = torch.tensor(Ztr[:5])
_mu, _lv = _m.encode(_x)
assert _mu.shape == (5, 8) and _lv.shape == (5, 8), 'the two heads must both be (n, zdim)'
assert _mu.min() < 0, 'mu should be free to go negative - did you put an activation on it?'
_out = _m.dec(_mu)
assert _out.shape == _x.shape, 'the decoder must return to the input width'
assert _out.min() < 0, ('the decoder output should be able to go negative - a ReLU or '
                        'Sigmoid on the last layer cannot represent standardised data')
_n = sum(p.numel() for p in _m.parameters())
print('CHECK 1 OK — %d parameters' % _n)

---
## 3 · TODO 2 — the sampling step, and the training loop

**The reparameterization trick.** We need a sample from $\mathcal{N}(\mu, \sigma^2)$ that
we can still backpropagate through. Draw the randomness from a fixed distribution instead
and transform it:

$$\epsilon \sim \mathcal{N}(0, I) \qquad z = \mu + \sigma \odot \epsilon$$

`logvar` holds $\log\sigma^2$, so $\sigma = \exp(\tfrac12\log\sigma^2)$.

If you return `mu` and skip the noise, everything still runs — and you have quietly built
an ordinary autoencoder. The check below catches exactly that.

In [ ]:
def reparameterize(mu, logvar):
    # ---- TODO 2a -----------------------------------------------------------
    # Return mu + sigma * eps, with eps drawn from a standard normal the same
    # shape as mu. Use torch.randn_like.
    raise SystemExit('TODO 2a: implement the reparameterization trick')
    # ---- end TODO 2a -------------------------------------------------------


def train_vae(Xtr, zdim, sigma2, epochs, seed, bs=128, lr=1e-3, verbose=False):
    model = VAE(Xtr.shape[1], zdim, seed=seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    Xt = torch.tensor(Xtr)
    history = []
    for ep in range(epochs):
        perm = torch.randperm(len(Xt), generator=g)
        tot = 0.0
        for i in range(0, len(Xt), bs):
            xb = Xt[perm[i:i + bs]]
            # ---- TODO 2b -------------------------------------------------------
            # Five lines, in the right order:
            #   xhat, mu, logvar = model(xb)
            #   recon, kl = vae_loss(xb, xhat, mu, logvar, sigma2)
            #   loss = -(recon - kl).mean()        # we MINIMISE the negative ELBO
            #   opt.zero_grad(); loss.backward(); opt.step()
            raise SystemExit('TODO 2b: write the five lines of the training step')
            # ---- end TODO 2b ---------------------------------------------------
            tot += float(loss) * len(xb)
        history.append(tot / len(Xt))
        if verbose and (ep + 1) % max(1, epochs // 5) == 0:
            print('   epoch %3d   -ELBO %10.3f' % (ep + 1, history[-1]))
    model.eval()
    return model, history

In [ ]:
# CHECK 2
torch.manual_seed(0)
_mu = torch.zeros(4000, 3); _lv = torch.zeros(4000, 3)
_z = reparameterize(_mu, _lv)
assert _z.shape == _mu.shape, 'reparameterize must keep the shape'
assert abs(float(_z.std()) - 1.0) < 0.1, (
    'with mu=0 and logvar=0 the samples should have sd about 1 - if it is 0 you '
    'returned mu and built a plain autoencoder')
_a = reparameterize(_mu, _lv); _b = reparameterize(_mu, _lv)
assert not torch.allclose(_a, _b), 'two calls must give two different samples'
_z2 = reparameterize(torch.zeros(4000, 3), torch.full((4000, 3), math.log(4.0)))
assert abs(float(_z2.std()) - 2.0) < 0.2, (
    'logvar holds log(sigma^2), so sigma = exp(0.5 * logvar)')
print('CHECK 2 OK — sd at logvar=0 is %.3f, at logvar=log 4 is %.3f'
      % (float(_z.std()), float(_z2.std())))

---
## 4 · TODO 3 — the loss, and the $\sigma^2$ you are choosing whether you mean to or not

The ELBO splits into two terms:

$$\mathcal{L}(x) = \underbrace{\mathbb{E}_{q}\left[\log p_\theta(x \mid z)\right]}_{\text{reconstruction}}
\;-\; \underbrace{D_{KL}\!\left(q_\phi(z \mid x)\,\|\,p(z)\right)}_{\text{regulariser}}$$

Your data is real-valued, so the decoder is Gaussian with an observation variance
$\sigma^2$:

$$\log p_\theta(x \mid z) = -\frac{1}{2}\sum_{j=1}^{d}
\left[\frac{(x_j - \hat{x}_j)^2}{\sigma^2} + \log\left(2\pi\sigma^2\right)\right]$$

and the KL has a closed form, so it needs no sampling at all:

$$-D_{KL} = \frac{1}{2}\sum_{j=1}^{J}\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

**Sum over dimensions, do not average.** Averaging over the $d$ features silently divides
the reconstruction term by $d$ and leaves the KL term alone, which changes the balance
between them by a factor of $d$. The check below catches it.

Notice that $\sigma^2$ is an *argument*. Writing a plain summed squared error instead
would be the same as fixing $\sigma^2 = 1/2$ — which is a claim about your data, and
almost certainly a wrong one. We start there, and TODO 5 comes back to it.

In [ ]:
def vae_loss(x, xhat, mu, logvar, sigma2):
    '''Returns (recon, kl), each a vector with one entry per row of x.

    recon is a LOG-LIKELIHOOD (bigger is better); kl is a DIVERGENCE (smaller is
    better). The ELBO is recon - kl.
    '''
    d = x.shape[1]
    # ---- TODO 3 ------------------------------------------------------------
    # recon = -0.5 * ( ((x - xhat)**2).sum(1) / sigma2
    #                  + d * math.log(2 * math.pi * sigma2) )
    # kl    = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1)
    raise SystemExit('TODO 3: return the Gaussian reconstruction term and the KL')
    # ---- end TODO 3 --------------------------------------------------------

In [ ]:
# CHECK 3
_x = torch.randn(7, 5); _xh = _x.clone()
_mu = torch.zeros(7, 3); _lv = torch.zeros(7, 3)
_r, _k = vae_loss(_x, _xh, _mu, _lv, 0.5)
assert _r.shape == (7,) and _k.shape == (7,), 'both terms are per row, shape (n,)'
assert torch.allclose(_k, torch.zeros(7), atol=1e-6), (
    'with mu=0 and logvar=0 the encoder IS the prior, so the KL must be exactly 0')
_expect = -0.5 * 5 * math.log(2 * math.pi * 0.5)
assert abs(float(_r[0]) - _expect) < 1e-4, (
    'a perfect reconstruction should leave only the log-normaliser, %.4f - did you '
    'average over features instead of summing?' % _expect)
_r2, _ = vae_loss(_x, _x + 1.0, _mu, _lv, 0.5)
assert float(_r2[0]) < float(_r[0]), 'a worse reconstruction must score lower'
_k2 = vae_loss(_x, _xh, torch.ones(7, 3), _lv, 0.5)[1]
assert float(_k2[0]) > 0, 'the KL must be positive once mu moves off zero'
print('CHECK 3 OK — perfect-reconstruction term %.4f, KL at the prior %.1e'
      % (float(_r[0]), float(_k[0])))

---
## 5 · Train it

Three seeds. One number from one seed is not a result — you need the spread to know
whether any difference you report is real.

In [ ]:
ZDIM = 16 if data_kind == 'image' else 8
EPOCHS = 120 if data_kind == 'image' else (300 if len(tr) < 1000 else 150)
SEEDS = [0, 1, 2]
SIGMA2_NAIVE = 0.5          # what a plain summed squared error assumes

models, histories = [], []
t0 = time.time()
for s in SEEDS:
    m, h = train_vae(Ztr, ZDIM, SIGMA2_NAIVE, EPOCHS, s, verbose=(s == 0))
    models.append(m); histories.append(h)
print('trained %d models in %.0f s' % (len(models), time.time() - t0))

plt.figure(figsize=(6.0, 3.0))
for s, h in zip(SEEDS, histories):
    plt.plot(h, lw=1.8, label='seed %d' % s)
plt.xlabel('epoch'); plt.ylabel('$-$ELBO on the training normals')
plt.title('Figure 1 — training'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

---
## 6 · TODO 4 — score, then find out whether it was worth it

The anomaly score is the **negative ELBO**: how surprised the model is. Average it over a
few noise draws, because a single $\epsilon$ makes the score noisy.

Remember what you are reporting. The ELBO is a *lower* bound on $\log p_\theta(x)$, so
$-$ELBO is an *upper* bound on the surprise. You are not measuring how improbable the
example is; you are measuring something that is at least as large.

In [ ]:
@torch.no_grad()
def anomaly_score(model, Xs, sigma2, n_samples=8, seed=0):
    '''Mean -ELBO per row, over n_samples draws of epsilon. Bigger = more anomalous.'''
    torch.manual_seed(seed)
    x = torch.tensor(Xs)
    total = torch.zeros(len(x))
    # ---- TODO 4 ------------------------------------------------------------
    # Loop n_samples times. Each time:
    #     xhat, mu, logvar = model(x)
    #     recon, kl = vae_loss(x, xhat, mu, logvar, sigma2)
    #     total += -(recon - kl)
    # Then return (total / n_samples).numpy()
    raise SystemExit('TODO 4: average the negative ELBO over n_samples noise draws')
    # ---- end TODO 4 --------------------------------------------------------

In [ ]:
# CHECK 4
_s = anomaly_score(models[0], Zte[:64], SIGMA2_NAIVE, n_samples=4)
assert isinstance(_s, np.ndarray) and _s.shape == (64,), 'return a numpy array, one score per row'
_a = anomaly_score(models[0], Zte[:64], SIGMA2_NAIVE, n_samples=1, seed=1)
_b = anomaly_score(models[0], Zte[:64], SIGMA2_NAIVE, n_samples=64, seed=1)
assert _a.std() > 0, 'the score should vary across rows'
assert np.corrcoef(_a, _b)[0, 1] > 0.9, 'more samples should not change the ranking much'
assert _b.std() > 0
print('CHECK 4 OK — mean score %.1f, sd across rows %.1f' % (_s.mean(), _s.std()))

### The baselines — written for you

Three things anyone could write in an afternoon, all fitted on the training normals only:

- **Mahalanobis distance** from a single Gaussian. One mean, one covariance.
- **PCA reconstruction error**, at the *same* latent size as your VAE, so the comparison
  is about the nonlinearity and not about capacity.
- **A one-class SVM** with an RBF kernel.

And one line you are not allowed to beat honestly: a **supervised** classifier fitted on
the validation split, which *has* labels. It is there to show what the labels would have
bought you if you had them.

In [ ]:
from sklearn.linear_model import LogisticRegression

def fit_baselines(Ztr, Zva, lab_va, zdim):
    out = {}
    C = np.cov(Ztr.T) + 1e-6 * np.eye(Ztr.shape[1])
    P, m = np.linalg.inv(C), Ztr.mean(0)
    out['Mahalanobis'] = lambda A: np.einsum('ij,jk,ik->i', A - m, P, A - m)
    p = PCA(n_components=zdim).fit(Ztr)
    out['PCA reconstruction'] = lambda A: ((A - p.inverse_transform(p.transform(A))) ** 2).sum(1)
    sv = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05).fit(Ztr)
    out['one-class SVM'] = lambda A: -sv.decision_function(A)
    lr = LogisticRegression(max_iter=5000, class_weight='balanced').fit(Zva, lab_va)
    out['supervised (uses labels!)'] = lambda A: lr.decision_function(A)
    return out

BASE = fit_baselines(Ztr, Zva, lab_va, ZDIM)
print('fitted:', ', '.join(BASE))

In [ ]:
# The comparison table. The VAE row is three seeds; report the spread.
vae_aucs = [roc_auc_score(lab_te, anomaly_score(m, Zte, SIGMA2_NAIVE, seed=s))
            for s, m in zip(SEEDS, models)]
rows = [('VAE, $-$ELBO', float(np.mean(vae_aucs)), float(np.std(vae_aucs)))]
for name, fn in BASE.items():
    rows.append((name, roc_auc_score(lab_te, fn(Zte)), 0.0))
rows.append(('chance', 0.5, 0.0))

print('%-28s %8s %8s' % ('method', 'AUC', 'sd'))
for n_, a_, s_ in rows:
    print('%-28s %8.4f %8.4f' % (n_.replace('$-$', '-'), a_, s_))
AUC_TABLE = rows

In [ ]:
# Figure 2 — ROC curves, written for you
plt.figure(figsize=(5.2, 5.0))
fpr, tpr, _ = roc_curve(lab_te, anomaly_score(models[0], Zte, SIGMA2_NAIVE, seed=0))
plt.plot(fpr, tpr, lw=2.4, color='#1F3864', label='VAE  %.3f' % vae_aucs[0])
for name, fn in BASE.items():
    f_, t_, _ = roc_curve(lab_te, fn(Zte))
    plt.plot(f_, t_, lw=1.4, ls='--', label='%s  %.3f' % (name, roc_auc_score(lab_te, fn(Zte))))
plt.plot([0, 1], [0, 1], color='grey', lw=1, ls=':')
plt.xlabel('false positive rate'); plt.ylabel('true positive rate')
plt.title('Figure 2 — ROC on the test split'); plt.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

### Not every anomaly is the same anomaly

A single pooled AUC averages over whatever mix of abnormal classes happens to be in your
test set. If one of them is easy and another is invisible, the pooled number hides it —
and in a clinical setting the invisible one is usually the one you cared about.

In [ ]:
# Figure 3 — per-class AUC, written for you
anom_classes = [c for c in sorted(set(y.tolist())) if c != normal and (y[te] == c).any()]
per_class = {}
for c in anom_classes:
    keep = (y[te] == normal) | (y[te] == c)
    aucs = [roc_auc_score((y[te][keep] == c).astype(int),
                          anomaly_score(m, Zte[keep], SIGMA2_NAIVE, seed=s))
            for s, m in zip(SEEDS, models)]
    per_class[class_names[c]] = (float(np.mean(aucs)), float(np.std(aucs)))

plt.figure(figsize=(6.2, 3.0))
ks = list(per_class)
plt.bar(range(len(ks)), [per_class[k][0] for k in ks],
        yerr=[per_class[k][1] for k in ks], color='#1F3864', width=.6, capsize=4)
plt.axhline(0.5, color='#9C2B2B', ls='--', lw=1.5)
plt.text(len(ks) - 0.5, 0.515, 'chance', color='#9C2B2B', fontsize=9, ha='right')
plt.xticks(range(len(ks)), ks, fontsize=9); plt.ylim(0, 1.05)
plt.ylabel('AUC vs normal'); plt.title('Figure 3 — detectability by anomaly class')
plt.tight_layout(); plt.show()
for k, (a_, s_) in per_class.items():
    print('%-32s AUC %.4f +/- %.4f' % (k, a_, s_))

---
## 7 · TODO 5 — the $\sigma^2$ you have been assuming

Everything above used `SIGMA2_NAIVE = 0.5`, which is what a plain summed squared error
assumes. Is that what your data says?

Fit it: run the training normals through the model, take the mean squared residual, and
you have the observation variance the model's own reconstructions imply. Then retrain
with that value and score again.

If $\hat\sigma^2$ comes out well below 0.5, the naive loss was multiplying your
reconstruction term by $1/(2\cdot 0.5) = 1$ when it should have multiplied it by
$1/(2\hat\sigma^2)$ — that is, it was over-weighting the KL term by a factor of
$0.5/\hat\sigma^2$.

In [ ]:
@torch.no_grad()
def fit_sigma2(model, Xtr_):
    '''The observation variance this model's own residuals imply.'''
    x = torch.tensor(Xtr_)
    # ---- TODO 5 ------------------------------------------------------------
    # Encode, decode the MEAN (no sampling - we want the model's best guess),
    # and return the mean squared residual as a float:
    #     mu, _ = model.encode(x)
    #     xhat  = model.dec(mu)
    #     return float(((x - xhat) ** 2).mean())
    raise SystemExit('TODO 5: return the mean squared residual on the training normals')
    # ---- end TODO 5 --------------------------------------------------------

In [ ]:
# CHECK 5
_s2 = fit_sigma2(models[0], Ztr)
assert isinstance(_s2, float) and _s2 > 0, 'return a single positive float'
assert _s2 < 5.0, 'a variance of %.2f on standardised data means something is wrong' % _s2
print('CHECK 5 OK — sigma^2 implied by the residuals: %.5f' % _s2)
print('            plain summed squared error assumes: %.3f' % SIGMA2_NAIVE)
print('            so the naive loss over-weighted the KL term by %.1fx'
      % (SIGMA2_NAIVE / _s2))

In [ ]:
# Retrain with the fitted sigma^2 and score again.
SIGMA2_FIT = float(np.mean([fit_sigma2(m, Ztr) for m in models]))
models2, aucs2 = [], []
for s in SEEDS:
    m2, _ = train_vae(Ztr, ZDIM, SIGMA2_FIT, EPOCHS, s)
    models2.append(m2)
    aucs2.append(roc_auc_score(lab_te, anomaly_score(m2, Zte, SIGMA2_FIT, seed=s)))

print('sigma^2 = %.3f (naive)   AUC %.4f +/- %.4f' % (SIGMA2_NAIVE, np.mean(vae_aucs),
                                                      np.std(vae_aucs)))
print('sigma^2 = %.5f (fitted)  AUC %.4f +/- %.4f' % (SIGMA2_FIT, np.mean(aucs2),
                                                      np.std(aucs2)))
print('change: %+.4f' % (np.mean(aucs2) - np.mean(vae_aucs)))

per_class2 = {}
for c in anom_classes:
    keep = (y[te] == normal) | (y[te] == c)
    a_ = [roc_auc_score((y[te][keep] == c).astype(int),
                        anomaly_score(m2, Zte[keep], SIGMA2_FIT, seed=s))
          for s, m2 in zip(SEEDS, models2)]
    per_class2[class_names[c]] = float(np.mean(a_))
for k in per_class:
    print('   %-30s %.4f -> %.4f' % (k, per_class[k][0], per_class2[k]))

---
## 8 · The verdict

Written for you. It compares your best VAE against the best thing you could have written
in two lines, and weighs the gap against your own seed spread.

In [ ]:
best_vae = max(float(np.mean(vae_aucs)), float(np.mean(aucs2)))
spread = max(float(np.std(vae_aucs)), float(np.std(aucs2)), 1e-4)
trivial = {k: roc_auc_score(lab_te, f(Zte)) for k, f in BASE.items()
           if 'supervised' not in k}
bn = max(trivial, key=trivial.get)
bb = trivial[bn]
sup = roc_auc_score(lab_te, BASE['supervised (uses labels!)'](Zte))

print('best VAE              %.4f   (seed spread +/- %.4f)' % (best_vae, spread))
print('best trivial baseline %.4f   (%s)' % (bb, bn))
print('supervised ceiling    %.4f   (had you been given labels)' % sup)
print()
if best_vae - bb > 2 * spread:
    print('VERDICT: the VAE beats every trivial baseline by %+.4f, which is more than '
          'twice\\n         your seed spread. The nonlinearity bought you something real.'
          % (best_vae - bb))
elif bb - best_vae > 2 * spread:
    print('VERDICT: the VAE LOSES to %s by %.4f, outside your seed spread.\\n'
          '         Report that. It is the result.' % (bn, bb - best_vae))
else:
    print('VERDICT: the gap to %s is %+.4f, inside twice your seed spread of %.4f.\\n'
          '         You cannot distinguish them with three seeds. Say so.'
          % (bn, best_vae - bb, spread))

In [ ]:
# Save what the memo needs.
summary = dict(
    track=TRACK, zdim=ZDIM, epochs=EPOCHS, seeds=SEEDS,
    n_train=int(len(tr)), n_test=int(len(te)),
    test_prevalence=round(float(lab_te.mean()), 4),
    vae_naive=[round(float(np.mean(vae_aucs)), 4), round(float(np.std(vae_aucs)), 4)],
    vae_fitted=[round(float(np.mean(aucs2)), 4), round(float(np.std(aucs2)), 4)],
    sigma2_fitted=round(SIGMA2_FIT, 5),
    sigma2_factor=round(SIGMA2_NAIVE / SIGMA2_FIT, 1),
    baselines={k: round(float(roc_auc_score(lab_te, f(Zte))), 4) for k, f in BASE.items()},
    per_class_naive={k: round(v[0], 4) for k, v in per_class.items()},
    per_class_fitted={k: round(v, 4) for k, v in per_class2.items()})
json.dump(summary, open('p5_results_%s.json' % TRACK, 'w'), indent=1)
print(json.dumps(summary, indent=1))

---
## 9 · The memo

One page, six questions. They are in §7 of the spec. Answer them with the numbers this
notebook printed, not with impressions.

1. What did your VAE score, and what did the best two-line baseline score? State the
   spread across seeds and say whether the gap is bigger than it.
2. Which anomaly class was hardest to detect, and why might that be — what would the VAE
   have to fail to reconstruct in order to flag it?
3. What $\sigma^2$ did your residuals imply? What was the naive loss assuming instead, and
   which direction did that push the balance between the two loss terms?
4. The supervised baseline used labels and the VAE did not. Given the gap, when would you
   still choose the unsupervised model in a real clinical setting?
5. You reported $-$ELBO, which is an upper bound on the surprise, not the surprise itself.
   Does that matter for *ranking* examples? Does it matter for reporting an absolute number?
6. Your training split contained no anomalies at all. Name one thing that could go wrong
   in deployment that this design would be unable to warn you about.